# Data Quality Checks on Novamind AI Historical Data

In [0]:
%sql
-- Checking for Nulls in the fields

SELECT * 
FROM novamind_ai.default.novamind_ai_historical
WHERE user_id IS NULL
    OR region IS NULL
    OR country IS NULL
    OR subscription_plan IS NULL
    OR months_subscribed IS NULL
    OR usage_frequency IS NULL
    OR feature_adoption IS NULL
    OR usage_frequency IS NULL
    OR feature_adoption IS NULL
    OR failed_payments IS NULL
    OR support_tickets IS NULL
    OR last_active_days_ago IS NULL
    OR prompt_volume IS NULL
    OR api_or_app IS NULL
    OR renewal_behavior IS NULL
    OR churned IS NULL

In [0]:
%sql
-- Checking for uniqueness of User ID

SELECT user_id, COUNT(*)
FROM novamind_ai.default.novamind_ai_historical
GROUP BY user_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Checking for integrity of data in region

SELECT 
  region, 
  COUNT(*) AS region_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS perc_of_total
FROM novamind_ai.default.novamind_ai_historical
GROUP BY region
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Checking for accepted values in subscription_plan

SELECT 
  subscription_plan, 
  COUNT(*) AS subscription_plan_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS perc_of_total
FROM novamind_ai.default.novamind_ai_historical
WHERE NOT IN ('Free', 'Pro', 'Business')
GROUP BY subscription_plan
ORDER BY subscription_plan;

In [0]:
%sql
-- Checking for freshness of data

SELECT 
    MIN(last_active_days_ago) AS most_recent,
    MAX(last_active_days_ago) AS least_recent,
    ROUND(AVG(last_active_days_ago), 2) AS avg_days_inactive
FROM novamind_ai.default.novamind_ai_historical;

# SQL EDA on NovaMind AI Historical data

In [0]:
%sql
-- Churn overview

SELECT 
  churned,
  COUNT(user_id) AS count,
  ROUND(COUNT(user_id) * 100.0 / SUM(COUNT(user_id)) OVER(),2) AS perc_of_count
FROM 
  novamind_ai.default.novamind_ai_historical
GROUP BY 
  churned
ORDER BY 
  churned

In [0]:
%sql
-- Regional breakdown of Churned users of NovaMind AI

SELECT
  region,
  COUNT(user_id) AS count_of_users,
  ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id),2) AS churn_rate
FROM
  novamind_ai.default.novamind_ai_historical
GROUP BY
  region
ORDER BY
  churn_rate DESC;

In [0]:
%sql

-- Breakdown of Subscription plan by the number of users and churn rate
SELECT
    subscription_plan,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    subscription_plan
ORDER BY
    churn_rate DESC

In [0]:
%sql
-- Breakdown of renewal behavior by the number of users and churn rate

SELECT
    renewal_behavior,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    renewal_behavior
ORDER BY
    churn_rate DESC

In [0]:
%sql
-- Which plan's skipped users have the highest churn rate?

SELECT
    renewal_behavior,
    subscription_plan,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    renewal_behavior, subscription_plan
ORDER BY
    renewal_behavior, churn_rate DESC

In [0]:
%sql
-- Breakdown of App/API by the number of users and churn rate

SELECT
    api_or_app,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    api_or_app
ORDER BY
    churn_rate DESC

In [0]:
%sql
-- Breakdown of App/API and subcription plan by the number of users and churn rate

SELECT
    api_or_app,
    subscription_plan,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    api_or_app,
    subscription_plan
ORDER BY
    churn_rate DESC

In [0]:
%sql
-- List the high risk users likely to churn
SELECT
    user_id,
    failed_payments,
    last_active_days_ago
FROM
    novamind_ai.default.novamind_ai_historical
WHERE
    failed_payments > 2
    AND last_active_days_ago > 20;

In [0]:
%sql
-- number of high risk users

In [0]:
%sql

-- Count the number of high risk users
SELECT
    COUNT(user_id) AS high_risk_users
FROM
    novamind_ai.default.novamind_ai_historical
WHERE
    failed_payments >=1
    AND last_active_days_ago > 15;

In [0]:
%sql
-- Distribution of failed_payments

-- from the results it looks like users who have faile dpoayments one time even have a high churn rate
SELECT
    failed_payments,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM novamind_ai.default.novamind_ai_historical
GROUP BY failed_payments
ORDER BY failed_payments

In [0]:
%sql
-- Distribution of last_active_days_ago in bands
SELECT
    CASE 
        WHEN last_active_days_ago <= 7 THEN '0-7 days'
        WHEN last_active_days_ago <= 14 THEN '8-14 days'
        WHEN last_active_days_ago <= 30 THEN '15-30 days'
        ELSE '30+ days'
    END AS inactivity_band,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM novamind_ai.default.novamind_ai_historical
GROUP BY inactivity_band
ORDER BY churn_rate DESC

In [0]:
%sql
-- the number of users who have failed payments and are inactive
SELECT
    churned,
    COUNT(user_id) AS users
FROM novamind_ai.default.novamind_ai_historical
WHERE
    failed_payments >= 1
    AND last_active_days_ago > 15
GROUP BY churned

In [0]:
%sql
-- Churn rate in African countries
SELECT
    country,
    subscription_plan,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
WHERE
    region = 'Africa'
GROUP BY
    country,
    subscription_plan
ORDER BY
    churn_rate DESC

In [0]:
%sql
-- Churn rate in African countries
SELECT
    country,
    subscription_plan,
    COUNT(user_id) AS total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_users,
    ROUND(SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(user_id), 2) AS churn_rate
FROM
    novamind_ai.default.novamind_ai_historical
GROUP BY
    country,
    subscription_plan
ORDER BY
    churn_rate DESC

# Modeling churn using Python

In [0]:
# load data from the Unity Catalog into Spark
df = spark.table('novamind_ai.default.novamind_ai_historical')

# convert to pandas for scikit-learn
pdf = df.toPandas()
print(f"Rows: {pdf.shape[0]}")
print(f"Columns: {pdf.shape[1]}")
print()
print(pdf.dtypes)

In [0]:
# Label Encoding — converting text columns to numbers
# The model is mathematical and cannot read text like "Renewed" or "Africa"
# We add new "_encoded" columns alongside the originals
# Originals are kept for Tableau visualisation later

from sklearn.preprocessing import LabelEncoder

# Columns that are text-based and need to be converted to numbers
label_cols = ["subscription_plan", "api_or_app", "renewal_behavior", "region"]

le = LabelEncoder()

for col_name in label_cols:
    pdf[col_name + "_encoded"] = le.fit_transform(pdf[col_name])

print("Encoding complete. New columns added:")
print([c for c in pdf.columns if "_encoded" in c])

In [0]:
# Train/Test Split
# We split the data so the model learns on one portion and is tested on another
# This prevents the model from "cheating" by being tested on data it already saw
from sklearn.model_selection import train_test_split

# Define features (what the model learns from) and target (what it predicts)
feature_cols = [
    "subscription_plan_encoded",
    "months_subscribed",
    "usage_frequency",
    "feature_adoption",
    "failed_payments",
    "support_tickets",
    "last_active_days_ago",
    "prompt_volume",
    "api_or_app_encoded",
    "renewal_behavior_encoded",
    "region_encoded"
]

X = pdf[feature_cols]  # Features
y = pdf["churned"]     # Target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")

## We are splitting 15,000 users into two groups — 80% to teach the model and 20% to test it

In [0]:
# Train the Random Forest Model
# We feed the training data into the model so it learns the patterns
# class_weight="balanced" handles the fact that only 13.8% of users churned
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees voting together
    max_depth=10,        # how deep each tree can go
    class_weight="balanced",  # prevents model from ignoring the minority churn class
    random_state=42      # ensures results are reproducible
)

model.fit(X_train, y_train)

print("Model training complete.")

## We are handing our 12,000 training users to 100 decision trees and saying "learn what makes a user churn." 

In [0]:
# Model Evaluation
# We test the model against the 3,000 users it has never seen before
# This tells us how well it learned to predict churn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Not Churned", "Churned"]))

print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(f"Correctly predicted staying:  {cm[0][0]}")
print(f"Predicted churn but stayed:   {cm[0][1]}")
print(f"Predicted stay but churned:   {cm[1][0]}")
print(f"Correctly predicted churned:  {cm[1][1]}")

In [0]:
# Feature Importance
# This tells us which factors the model relied on most to predict churn
# The higher the score, the more influential that feature was
import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("=== Feature Importance Rankings ===")
print(importance_df.to_string(index=False))

## Running churn model on the current users

In [0]:
# Load Current Users from Unity Catalog
# These are active users with no known churn outcome
# The model will score each one with a churn probability
df_current = spark.table("novamind_ai.default.novamind_ai_current_users")
pdf_current = df_current.toPandas()

print(f"Current Users: {pdf_current.shape[0]} rows")
print(f"Columns: {pdf_current.shape[1]}")
pdf_current.head()

In [0]:
# Apply the same encoding we did on historical data
# The model expects the same format it was trained on
for col_name in label_cols:
    pdf_current[col_name + "_encoded"] = le.fit_transform(pdf_current[col_name])

print("Encoding complete.")

In [0]:
# Run predictions on current users
# No churned column here — the model predicts it for us

def risk_segment(prob):
    if prob >= 0.75:
        return 'High'
    elif prob >= 0.4:
        return 'Medium'
    else:
        return 'Low'

X_current = pdf_current[feature_cols]

pdf_current["churn_probability"] = model.predict_proba(X_current)[:, 1]
pdf_current["churn_prediction"] = model.predict(X_current)
pdf_current["risk_segment"] = pdf_current["churn_probability"].apply(risk_segment)

print("=== Current Users Risk Distribution ===")
print(pdf_current["risk_segment"].value_counts())
print()
print("=== Predicted Churn Count ===")
print(pdf_current["churn_prediction"].value_counts())

In [0]:
# Prepare Export Table
# Select only the columns needed for Tableau visualisation
# Dropping encoded columns — Tableau needs readable text not numbers

export_cols = [
    "user_id",
    "country",
    "region",
    "subscription_plan",
    "months_subscribed",
    "usage_frequency",
    "feature_adoption",
    "failed_payments",
    "support_tickets",
    "last_active_days_ago",
    "prompt_volume",
    "api_or_app",
    "renewal_behavior",
    "churn_probability",
    "churn_prediction",
    "risk_segment"
]

export_df = pdf_current[export_cols].copy()

# Round churn probability to 2 decimal places for readability
export_df["churn_probability"] = export_df["churn_probability"].round(2)

print(f"Export shape: {export_df.shape}")

In [0]:
# Convert Pandas dataframe back to Spark
spark_export = spark.createDataFrame(export_df)

# Verify it converted correctly
print(f"Rows to save: {spark_export.count()}")
spark_export.printSchema()

In [0]:
# Set the catalog and schema explicitly first
spark.sql("USE CATALOG novamind_ai")
spark.sql("USE SCHEMA default")

# Write to catalog
spark_export.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("novamind_ai.default.novamind_ai_predictions")

# Verify it saved
verify = spark.sql("SELECT COUNT(*) as total_rows FROM novamind_ai.default.novamind_ai_predictions")
display(verify)

print("Done — check your catalog now.")

# Semi-Automating the churn model for new data

In [0]:
# ============================================================
# NOVAMIND AI — CHURN PREDICTION PIPELINE
# Configuration — all settings live here
# To update the pipeline, only this cell needs to change
# ============================================================

config = {
    # Unity Catalog table paths
    "historical_table":   "novamind_ai.default.novamind_ai_historical",
    "current_users_table":"novamind_ai.default.novamind_ai_current_users",
    "predictions_table":  "novamind_ai.default.novamind_ai_predictions",

    # Model settings
    "n_estimators":       100,
    "max_depth":          10,
    "test_size":          0.2,
    "random_state":       42,

    # Risk thresholds
    "high_risk_threshold":   0.70,
    "medium_risk_threshold": 0.40,

    # Feature columns
    "feature_cols": [
        "subscription_plan_encoded",
        "months_subscribed",
        "usage_frequency",
        "feature_adoption",
        "failed_payments",
        "support_tickets",
        "last_active_days_ago",
        "prompt_volume",
        "api_or_app_encoded",
        "renewal_behavior_encoded",
        "region_encoded"
    ],

    # Columns to encode
    "label_cols": [
        "subscription_plan",
        "api_or_app",
        "renewal_behavior",
        "region"
    ],

    # Columns to export to Tableau
    "export_cols": [
        "user_id", "country", "region", "subscription_plan",
        "months_subscribed", "usage_frequency", "feature_adoption",
        "failed_payments", "support_tickets", "last_active_days_ago",
        "prompt_volume", "api_or_app", "renewal_behavior",
        "churn_probability", "churn_prediction", "risk_segment"
    ]
}

print("Configuration loaded.")
print(f"Historical table:    {config['historical_table']}")
print(f"Current users table: {config['current_users_table']}")
print(f"Predictions table:   {config['predictions_table']}")

In [0]:
# ============================================================
# DATA QUALITY CHECKS FUNCTION
# Runs all quality checks on any table passed into it
# Prints a clear PASSED or FAILED for each check
# ============================================================

def run_quality_checks(df, table_name):
    print(f"")
    print(f"============================================================")
    print(f"DATA QUALITY CHECKS — {table_name}")
    print(f"============================================================")

    passed = 0
    failed = 0

    # CHECK 1 — Row Count
    row_count = df.count()
    print(f"\n CHECK 1 — Row Count")
    print(f" Total rows: {row_count}")
    if row_count > 0:
        print(f" ✅ PASSED")
        passed += 1
    else:
        print(f" ❌ FAILED — Table is empty")
        failed += 1

    # CHECK 2 — Null Values
    from pyspark.sql.functions import isnull, sum as spark_sum
    print(f"\n CHECK 2 — Null Values")
    null_counts = df.select([
        spark_sum(isnull(c).cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    nulls_found = {k: v for k, v in null_counts.items() if v > 0}
    if not nulls_found:
        print(f" ✅ PASSED — No null values found")
        passed += 1
    else:
        print(f" ❌ FAILED — Nulls found: {nulls_found}")
        failed += 1

    # CHECK 3 — Duplicate User IDs
    from pyspark.sql.functions import count
    print(f"\n CHECK 3 — Duplicate User IDs")
    duplicates = df.groupBy("user_id") \
        .count() \
        .filter("count > 1") \
        .count()
    if duplicates == 0:
        print(f" ✅ PASSED — No duplicate user IDs")
        passed += 1
    else:
        print(f" ❌ FAILED — {duplicates} duplicate user IDs found")
        failed += 1

    # CHECK 4 — Accepted Values: subscription_plan
    from pyspark.sql.functions import col
    print(f"\n CHECK 4 — Accepted Values: subscription_plan")
    valid_plans = ["Free", "Pro", "Business"]
    invalid_plans = df.filter(~col("subscription_plan").isin(valid_plans)).count()
    if invalid_plans == 0:
        print(f" ✅ PASSED — All values in {valid_plans}")
        passed += 1
    else:
        print(f" ❌ FAILED — {invalid_plans} invalid subscription_plan values")
        failed += 1

    # CHECK 5 — Accepted Values: renewal_behavior
    print(f"\n CHECK 5 — Accepted Values: renewal_behavior")
    valid_renewal = ["Renewed", "Downgraded", "Skipped"]
    invalid_renewal = df.filter(~col("renewal_behavior").isin(valid_renewal)).count()
    if invalid_renewal == 0:
        print(f" ✅ PASSED — All values in {valid_renewal}")
        passed += 1
    else:
        print(f" ❌ FAILED — {invalid_renewal} invalid renewal_behavior values")
        failed += 1

    # CHECK 6 — Accepted Values: region
    print(f"\n CHECK 6 — Accepted Values: region")
    valid_regions = ["Africa", "Europe", "Asia", 
                     "North America", "Latin America", "Oceania"]
    invalid_regions = df.filter(~col("region").isin(valid_regions)).count()
    if invalid_regions == 0:
        print(f" ✅ PASSED — All values in {valid_regions}")
        passed += 1
    else:
        print(f" ❌ FAILED — {invalid_regions} invalid region values")
        failed += 1

    # CHECK 7 — Numeric Ranges
    from pyspark.sql.functions import min as spark_min, max as spark_max
    print(f"\n CHECK 7 — Numeric Ranges")
    range_issues = df.filter(
        (col("usage_frequency") < 0) |
        (col("failed_payments") < 0) |
        (col("months_subscribed") < 1) |
        (col("prompt_volume") < 0) |
        (col("feature_adoption") < 1) |
        (col("feature_adoption") > 10)
    ).count()
    if range_issues == 0:
        print(f" ✅ PASSED — All numeric values within expected ranges")
        passed += 1
    else:
        print(f" ❌ FAILED — {range_issues} rows with out of range values")
        failed += 1

    # SUMMARY
    print(f"")
    print(f"============================================================")
    print(f"QUALITY CHECK SUMMARY — {table_name}")
    print(f"Passed: {passed} | Failed: {failed} | Total: {passed + failed}")
    if failed == 0:
        print(f"✅ ALL CHECKS PASSED — Data is ready for processing")
    else:
        print(f"⚠️  {failed} CHECK(S) FAILED — Review before proceeding")
    print(f"============================================================")

print("Data quality function loaded.")

In [0]:
# ============================================================
# FEATURE ENGINEERING & ENCODING FUNCTION
# Converts text columns to numbers for the model
# Works on any dataframe passed into it
# ============================================================

from sklearn.preprocessing import LabelEncoder
import pandas as pd

def encode_features(df, config):
    print("\n============================================================")
    print("FEATURE ENGINEERING & ENCODING")
    print("============================================================")

    # Convert Spark dataframe to Pandas
    pdf = df.toPandas()
    print(f"\n Converted to Pandas — {pdf.shape[0]} rows, {pdf.shape[1]} columns")

    # Apply Label Encoding to each categorical column
    le = LabelEncoder()
    for col_name in config["label_cols"]:
        pdf[col_name + "_encoded"] = le.fit_transform(pdf[col_name])
        print(f" ✅ Encoded: {col_name} → {col_name}_encoded")

    print(f"\n Feature engineering complete.")
    print(f" Total columns now: {pdf.shape[1]}")

    return pdf, le

print("Feature engineering function loaded.")

In [0]:
# ============================================================
# MODEL TRAINING FUNCTION
# Trains the Random Forest on historical data only
# Returns the trained model and evaluation metrics
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

def train_model(pdf, config):
    print("\n============================================================")
    print("MODEL TRAINING")
    print("============================================================")

    # Define features and target
    X = pdf[config["feature_cols"]]
    y = pdf["churned"]

    # Train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config["test_size"],
        random_state=config["random_state"],
        stratify=y
    )
    print(f"\n Training set: {X_train.shape[0]} rows")
    print(f" Test set:     {X_test.shape[0]} rows")

    # Train Random Forest
    print(f"\n Training Random Forest model...")
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        class_weight="balanced",
        random_state=config["random_state"]
    )
    model.fit(X_train, y_train)

    # Evaluate
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc  = roc_auc_score(y_test, y_proba)

    print(f"\n ✅ Model training complete")
    print(f" Accuracy:  {accuracy:.4f}")
    print(f" ROC-AUC:   {roc_auc:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned'])}")

    return model

print("Model training function loaded.")

In [0]:
# ============================================================
# SCORING & RISK SEGMENTATION FUNCTION
# Takes current users and scores them with the trained model
# Assigns churn probability and risk segment to each user
# ============================================================

def score_users(df_current, model, config):
    print("\n============================================================")
    print("SCORING CURRENT USERS")
    print("============================================================")

    # Encode current users the same way as training data
    pdf_current, _ = encode_features(df_current, config)

    # Run predictions
    X_current = pdf_current[config["feature_cols"]]
    pdf_current["churn_probability"] = model.predict_proba(X_current)[:, 1].round(4)
    pdf_current["churn_prediction"]  = model.predict(X_current)

    # Assign risk segments
    def risk_segment(prob):
        if prob >= config["high_risk_threshold"]:
            return "High Risk"
        elif prob >= config["medium_risk_threshold"]:
            return "Medium Risk"
        else:
            return "Low Risk"

    pdf_current["risk_segment"] = pdf_current["churn_probability"].apply(risk_segment)

    # Summary
    total         = len(pdf_current)
    high_risk     = (pdf_current["risk_segment"] == "High Risk").sum()
    medium_risk   = (pdf_current["risk_segment"] == "Medium Risk").sum()
    low_risk      = (pdf_current["risk_segment"] == "Low Risk").sum()
    pred_churned  = (pdf_current["churn_prediction"] == 1).sum()

    print(f"\n Total users scored:    {total}")
    print(f" Predicted to churn:    {pred_churned} ({round(pred_churned/total*100, 1)}%)")
    print(f"\n Risk Segments:")
    print(f"   🔴 High Risk:         {high_risk} ({round(high_risk/total*100, 1)}%)")
    print(f"   🟡 Medium Risk:       {medium_risk} ({round(medium_risk/total*100, 1)}%)")
    print(f"   🟢 Low Risk:          {low_risk} ({round(low_risk/total*100, 1)}%)")

    return pdf_current

print("Scoring function loaded.")

In [0]:
# ============================================================
# SAVE PREDICTIONS FUNCTION
# Saves scored users back to Unity Catalog
# Tableau reads from this table automatically on refresh
# ============================================================

def save_predictions(pdf_scored, config):
    print("\n============================================================")
    print("SAVING PREDICTIONS TO UNITY CATALOG")
    print("============================================================")

    # Select only export columns
    export_df = pdf_scored[config["export_cols"]].copy()

    # Convert back to Spark and save as Delta table
    spark_export = spark.createDataFrame(export_df)

    spark.sql(f"USE CATALOG novamind_ai")
    spark.sql(f"USE SCHEMA default")

    spark_export.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(config["predictions_table"])

    # Verify save
    saved_count = spark.table(config["predictions_table"]).count()

    print(f"\n ✅ Predictions saved successfully")
    print(f" Table:  {config['predictions_table']}")
    print(f" Rows:   {saved_count}")
    print(f"\n Tableau will reflect new results on next refresh.")

print("Save predictions function loaded.")

In [0]:
# ============================================================
# NOVAMIND AI -- CHURN PREDICTION PIPELINE
# RUN THIS CELL TO EXECUTE THE FULL PIPELINE
# ============================================================

import time
start_time = time.time()

print("============================================================")
print("   NOVAMIND AI -- CHURN PREDICTION PIPELINE STARTING")
print("============================================================")

# -- STEP 1 -- LOAD DATA -------------------------------------
print("\n STEP 1 -- Loading Data...")
df_historical = spark.table(config["historical_table"])
df_current    = spark.table(config["current_users_table"])
print(f" Historical users: {df_historical.count()}")
print(f" Current users:    {df_current.count()}")

# -- STEP 2 -- DATA QUALITY CHECKS ---------------------------
print("\n STEP 2 -- Running Data Quality Checks...")
run_quality_checks(df_historical, "Historical Data")
run_quality_checks(df_current,    "Current Users")

# -- STEP 3 -- ENCODE HISTORICAL DATA ------------------------
print("\n STEP 3 -- Encoding Historical Data...")
pdf_historical, le = encode_features(df_historical, config)

# -- STEP 4 -- TRAIN MODEL ------------------------------------
print("\n STEP 4 -- Training Model...")
model = train_model(pdf_historical, config)

# -- STEP 5 -- SCORE CURRENT USERS ---------------------------
print("\n STEP 5 -- Scoring Current Users...")
pdf_scored = score_users(df_current, model, config)

# -- STEP 6 -- SAVE PREDICTIONS ------------------------------
print("\n STEP 6 -- Saving Predictions...")
save_predictions(pdf_scored, config)

# -- PIPELINE COMPLETE ---------------------------------------
end_time = time.time()
duration = round(end_time - start_time, 2)

print("\n============================================================")
print("   PIPELINE COMPLETE")
print(f"   Total runtime: {duration} seconds")
print("============================================================")
print("\n Next steps:")
print(" 1. Open Tableau Desktop")
print(" 2. Go to Data -> Refresh Data Source")
print(" 3. Dashboard will reflect latest predictions")